# Regression Modeling Workflow

This notebook summarizes the regression modeling portion of my thesis project, which tested whether daily precipitation amounts in South Carolina could be predicted using historical climate variables.

## Sections

1. Project Setup  
2. Data Loading  
3. Data Preparation  
4. Monthly Regression Modeling  
5. Walk-Forward Validation  
6. Random Forest Baseline  
7. Feature Engineering Experiments  
8. Random Forest Hyperparameter Tuning  
9. XGBoost Regression  
10. CatBoost Regression  
11. Final Regression Results  

The regression models showed limited predictive performance overall, which supported reframing the problem as a classification task in the next notebook.

In [ ]:
# import necessary libraries
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
import numpy as np
import glob

### Preprocessing 2024 Test data

In [ ]:
# Concatenate individual 2024 grid-location files into a single dataset

def files_concat():

    # Directory containing all 2024 grid-location CSV files
    directory = "<PATH_TO_2024_CSV_FILES>"

    # Get a list of all CSV files
    csv_files = glob.glob(os.path.join(directory, "*.csv"))

    # Read each CSV file into a DataFrame
    list_of_dfs = [pd.read_csv(file, encoding="latin-1") for file in csv_files]

    # Merge all DataFrames into one
    merged_df = pd.concat(list_of_dfs, ignore_index=True)

    # Save the merged dataset
    merged_df.to_csv("<OUTPUT_FILE_PATH>", index=False)

# Run the function
files_concat()

In [ ]:
# Split the merged 2024 dataset into monthly datasets
def split_data_by_month(file_path, output_dir):

    # Read the merged 2024 dataset
    df = pd.read_csv(file_path)

    # Create the output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)

    # Split the data by month
    for month in range(1, 13):

        # Filter rows for the current month
        month_df = df[df["Month"] == month]

        # Create the output file path
        output_file = os.path.join(output_dir, f"month_{month}.csv")

        # Save the monthly dataset
        month_df.to_csv(output_file, index=False)

# Input merged dataset
train_file = "<PATH_TO_MERGED_2024_DATASET>"

# Directory to save monthly datasets
train_output_dir = "<OUTPUT_DIRECTORY_FOR_MONTHLY_FILES>"

# Run the function
split_data_by_month(train_file, train_output_dir)

### Loading 2000-2023 Training data

In [ ]:
# Load each month's dataset into a dictionary of DataFrames

# Directory containing monthly training datasets
train_dir = "<PATH_TO_MONTHLY_TRAINING_DATA>"

# Store each month's DataFrame
monthly_dfs = {}

# Load each monthly CSV file
for month in range(1, 13):

    # Create the file path
    file_path = os.path.join(train_dir, f"month_{month}.csv")

    # Read the dataset
    monthly_dfs[month] = pd.read_csv(file_path)

### Data preprocessing & Exploratory Data Analysis

In [ ]:
# Calculate the percentage of missing values for each month

def check_missing_values(monthly_dfs):

    # Store missing-value percentages for each month
    missing_values = {}

    # Calculate missing-value percentage for each month's dataset
    for month, df in monthly_dfs.items():

        missing_percentage = df.isnull().sum() / len(df) * 100

        missing_values[month] = missing_percentage

    return pd.DataFrame(missing_values)


# Calculate missing-value percentages
missing_df = check_missing_values(monthly_dfs)

# Display January's missing values
missing_df[1]

In [ ]:
# Create heatmaps showing missing values for each month's training dataset

def plot_missing_values_heatmap(df, month):

    # Set the figure size
    plt.figure(figsize=(10, 6))

    # Visualize missing values
    sns.heatmap(df.isnull(), cbar=False, cmap="viridis")

    # Set the plot title
    plt.title(f"Missing Values Heatmap - Month {month}")

    plt.show()


# Generate a heatmap for each month's dataset
for month, df in monthly_dfs.items():
    plot_missing_values_heatmap(df, month)

In [ ]:
# Median imputation for missing data

for month, df in monthly_dfs.items():
    # Fill missing values in the 'etr(mm)' column with the median of that column.
    df['etr(mm)'] = df['etr(mm)'].fillna(df['etr(mm)'].median())

# Display remaining missing data    
monthly_dfs[12].isnull().sum()    

In [ ]:
# Linear plots showing monthly average precipitation trends (2000–2023)

# Monthly DataFrames ordered from January to December
dfs = [monthly_dfs[i] for i in range(1, 13)]
month_names = ["January", "February", "March", "April", "May", "June", "July", "August", "September", "October", "November", "December"]

# Set up a 3-column layout
fig, axes = plt.subplots(4, 3, figsize=(18, 12))  
axes = axes.flatten() 

for i, df in enumerate(dfs):
    # Calculate the average precipitation for each year
    monthly_avg = df.groupby(df['Year'])['pr(mm)'].mean()

    # Select the subplot for the current month
    ax = axes[i]
    
    # Plot the yearly averages
    ax.plot(monthly_avg.index, monthly_avg.values, marker='o', linestyle='-', label='Avg Precip')
   
    # Use the years to fit a linear trend
    trend = pd.Series(monthly_avg.index)

    # Fit a linear trend line
    z = np.polyfit(trend, monthly_avg.values, 1)
    
    # Create the trend line
    p = np.poly1d(z)
    
    # Plot the trend line
    ax.plot(monthly_avg.index, p(monthly_avg.index), linestyle='--', linewidth=3, color='orange', label='Precipitation Trend')

    ax.set_xlabel("Year")
    ax.set_ylabel("Average Precip (mm)")
    ax.set_title(f"Precipitation Trend for {month_names[i]}")
    ax.grid(True)
    ax.legend()

# Adjust subplot spacing
plt.tight_layout()
plt.show()


In [ ]:
# Calculate Pearson correlation matrices for each month's dataset

# Store Pearson correlation matrices
pc = {}
for month, df in monthly_dfs.items():
    
    # Calculate the Pearson correlation matrix
    pearson = df.corr(method='pearson', min_periods=1)
    pc[month] = pearson
# Display January's correlation matrix 
pc[1]

In [ ]:
# Display January's Pearson Correlation Heatmap

# Drop the 'Month' column
corr_matrix = pc[1].drop(columns='Month', errors='ignore')

# Drop the 'Month' row
corr_matrix = corr_matrix.drop(index='Month', errors='ignore')


# Plot the heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("January Pearson Correlation Heatmap")
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

# Calculate Variance Inflation Factor (VIF) for the selected features
features = [
    "etr(mm)",
    "rmax(%)",
    "sph(kg/kg)",
    "srad(Wm-2)",
    "vs(mps)",
    "th(DegreesClockwisefromnorth)"
]

# Use January's dataset
X = monthly_dfs[1][features]

# Add a constant term for VIF calculation
X_with_const = add_constant(X)

# Calculate VIF for each feature
vif_data = pd.DataFrame({
    "Feature": X_with_const.columns,
    "VIF": [
        variance_inflation_factor(X_with_const.values, i)
        for i in range(X_with_const.shape[1])
    ]
})

# Display January's VIF values
vif_data

In [ ]:
# Create a copy of the monthly datasets
monthly_dfs_poly = monthly_dfs.copy()

# Sort each dataset by climate region and date before feature engineering
for month in range(1, 13):

    df = monthly_dfs_poly[month]

    # Sort by climate region and chronological order
    df = df.sort_values(["ClimateRegion", "Year", "Month", "Day"])

    # Create a date column
    df["Date"] = pd.to_datetime(df[["Year", "Month", "Day"]])

    # Exclude 2024 data
    df = df[df["Year"] != 2024]

    # Save the processed dataset
    df.to_csv("<OUTPUT_FILE_PATH>", index=False)

    # Update the dictionary
    monthly_dfs_poly[month] = df

In [ ]:
# Reset index for each month
for month in range(1, 13):
    monthly_dfs_poly[month] = monthly_dfs_poly[month].reset_index(drop=True)
# Display January's dataset
monthly_dfs_poly[1]

In [ ]:
# Count observations in each climate region for January
category_counts = monthly_dfs_poly[1].groupby('ClimateRegion')['Day'].count()
category_counts

## Model Training and Validation

This section compares regression models for daily precipitation prediction using walk-forward validation. Models tested included Linear Regression, KNN, Random Forest, XGBoost, and CatBoost.

The best-performing regression models were Random Forest, XGBoost, and CatBoost. Additional experiments tested monthly vs. seasonal modeling, lag features, rolling averages, and hyperparameter tuning.

Key findings:
- Monthly grouping performed better than seasonal grouping.
- Lag and rolling-average features did not meaningfully improve Random Forest performance.
- Lag features slightly improved CatBoost performance.
- Overall regression performance remained limited, supporting the decision to reframe the problem as a classification task.

### Random Forest Regression Baseline

This section evaluates Random Forest regression using walk-forward validation. Multiple training/testing window sizes were tested for each month, and the configuration with the highest average testing R² was selected.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
import pandas as pd
import numpy as np

def split_data(df, start, training_window, forecast_horizon):
    """Split time-series data into training and testing windows."""
    train_df = df.iloc[start : start + training_window]
    test_df = df.iloc[start + training_window : start + training_window + forecast_horizon]
    return train_df, test_df


def prepare_features(train_df, test_df, target_col="pr(mm)", date_col="Date"):
    """Separate features/target, one-hot encode categorical columns, and align train/test columns."""
    X_train = train_df.drop(columns=[target_col, date_col], errors="ignore")
    y_train = train_df[target_col]

    X_test = test_df.drop(columns=[target_col, date_col], errors="ignore")
    y_test = test_df[target_col]

    X_train = pd.get_dummies(X_train, columns=["ClimateRegion"], prefix="Region")
    X_test = pd.get_dummies(X_test, columns=["ClimateRegion"], prefix="Region")

    X_train, X_test = X_train.align(X_test, join="left", axis=1, fill_value=0)

    return X_train, X_test, y_train, y_test


def walk_forward_evaluation(df, training_window, forecast_horizon):
    """Run walk-forward validation using Random Forest regression."""
    train_r2s = []
    test_r2s = []

    for start in range(0, len(df) - training_window - forecast_horizon, forecast_horizon):
        train_df, test_df = split_data(df, start, training_window, forecast_horizon)
        X_train, X_test, y_train, y_test = prepare_features(train_df, test_df)

        model = RandomForestRegressor(n_estimators=100, random_state=42)
        model.fit(X_train, y_train)

        train_r2s.append(r2_score(y_train, model.predict(X_train)))
        test_r2s.append(r2_score(y_test, model.predict(X_test)))

    return np.mean(train_r2s), np.mean(test_r2s)


def nested_walk_forward(df, fold_sizes=[3, 5, 10], partition_ratios=None):
    """Test multiple training/testing window sizes and return validation results."""
    if partition_ratios is None:
        partition_ratios = [(i, i + 1) for i in range(2, 12)]

    results = []
    n_rows = len(df)

    for fold_size in fold_sizes:
        fold_length = n_rows // fold_size

        for num, denom in partition_ratios:
            train_size = (num * fold_length) // denom
            test_size = fold_length - train_size

            if test_size <= 0:
                continue

            avg_train_r2, avg_test_r2 = walk_forward_evaluation(
                df,
                training_window=train_size,
                forecast_horizon=test_size
            )

            results.append({
                "fold_size": fold_size,
                "split": f"{num}/{denom}-{denom-num}/{denom}",
                "train_size": train_size,
                "test_size": test_size,
                "avg_train_r2": avg_train_r2,
                "avg_test_r2": avg_test_r2
            })

    return pd.DataFrame(results)

In [ ]:
monthly_results = {}
best_configs = {}

for month in range(1, 13):
    results_df = nested_walk_forward(monthly_dfs_poly[month])

    monthly_results[month] = results_df
    best_configs[month] = results_df.loc[results_df["avg_test_r2"].idxmax()]

best_config_summary = pd.DataFrame(best_configs).T
best_config_summary

The best-performing Random Forest configuration varied by month. This baseline was later compared with tuned Random Forest, XGBoost, and CatBoost models.

### Feature Engineering Experiment

Lag and rolling-average features were tested to determine whether recent climate patterns improved Random Forest regression performance. Features included lag-1 values and 7-, 14-, and 28-day rolling averages grouped by climate region.

In [ ]:
def add_rolling_features(monthly_data, features, windows=[7, 14, 28]):
    monthly_data_with_features = {}

    for month, df in monthly_data.items():
        df = df.copy()

        for feature in features:
            for window in windows:
                df[f"{feature}_rolling{window}"] = (
                    df.groupby("ClimateRegion")[feature]
                    .transform(lambda x: x.shift(1).rolling(window).mean())
                )

                df[f"{feature}_rolling{window}"] = df[f"{feature}_rolling{window}"].fillna(
                    df[f"{feature}_rolling{window}"].mean()
                )

        monthly_data_with_features[month] = df

    return monthly_data_with_features


main_features = ["etr(mm)", "rmax(%)", "sph(kg/kg)", "srad(Wm-2)", "vs(mps)"]

rolling_monthly = add_rolling_features(
    monthly_dfs_poly,
    features=main_features,
    windows=[7, 14, 28]
)

In [ ]:
monthly_results_rf_rolling = {}

for month in range(1, 13):
    config = best_configs_rf[month]

    train_r2s, test_r2s = perform_walk_forward_rf(
        rolling_monthly[month],
        training_window=config["train_size"],
        forecast_horizon=config["test_size"],
        best_params=best_params_per_month_rf[month]
    )

    monthly_results_rf_rolling[month] = {
        "train_r2": np.mean(train_r2s),
        "test_r2": np.mean(test_r2s)
    }

rf_rolling_summary = pd.DataFrame(monthly_results_rf_rolling).T
rf_rolling_summary

The lag and rolling-average features did not meaningfully improve Random Forest performance, so the final regression comparison used the original monthly feature set.

## Random Forest Hyperparameter Tuning

After establishing a baseline Random Forest model, I optimized the model using randomized hyperparameter search with time-series cross-validation.

The parameters evaluated included:

- Number of trees (`n_estimators`)
- Maximum tree depth (`max_depth`)
- Minimum samples required to split a node (`min_samples_split`)
- Minimum samples required at each leaf (`min_samples_leaf`)
- Number of features considered at each split (`max_features`)

In [ ]:
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV

param_grid_rf = {
    "n_estimators": [100, 200, 500, 700, 1000],
    "max_depth": [5, 10, 15, None],
    "min_samples_split": [2, 5, 10, 15],
    "min_samples_leaf": [1, 2, 5, 10],
    "max_features": ["sqrt", "log2", 0.5, 0.8]
}

tscv = TimeSeriesSplit(n_splits=5)

best_params_per_month_rf = {}

for month in range(1, 13):

    df = monthly_dfs_poly[month]

    X = df.drop(columns=["pr(mm)", "Date"], errors="ignore")
    y = df["pr(mm)"]

    X = pd.get_dummies(
        X,
        columns=["ClimateRegion"],
        prefix="Region"
    )

    rf = RandomForestRegressor(random_state=42)

    search = RandomizedSearchCV(
        estimator=rf,
        param_distributions=param_grid_rf,
        cv=tscv,
        n_iter=60,
        random_state=42,
        n_jobs=-1,
        verbose=0
    )

    search.fit(X, y)

    best_params_per_month_rf[month] = search.best_params_

The optimized hyperparameters from each month were then used during walk-forward validation to evaluate whether tuning improved predictive performance over the baseline Random Forest model.

In [ ]:
monthly_results_rf = {}

for month in range(1, 13):

    config = best_configs[month]

    train_r2s, test_r2s = perform_walk_forward_rf(
        monthly_dfs_poly[month],
        training_window=int(config["train_size"]),
        forecast_horizon=int(config["test_size"]),
        best_params=best_params_per_month_rf[month]
    )

    monthly_results_rf[month] = {
        "Train R²": np.mean(train_r2s),
        "Test R²": np.mean(test_r2s)
    }

rf_results_summary = pd.DataFrame(monthly_results_rf).T
rf_results_summary

### Results

Hyperparameter tuning produced modest improvements in model performance but did not substantially improve the overall predictive ability of Random Forest regression. The tuned model remained less effective than the final XGBoost regression model.

## XGBoost Regression

XGBoost was tested after Random Forest because gradient boosting can capture complex, non-linear relationships in climate data. Hyperparameter tuning was performed using randomized search with time-series cross-validation.

In [ ]:
from xgboost import XGBRegressor

param_grid_xgb = {
    "n_estimators": [50, 75, 100, 150, 200],
    "max_depth": [2, 3, 4, 6, 8, 10],
    "learning_rate": [0.01, 0.05, 0.1, 0.2, 0.3],
    "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
    "gamma": [0, 0.1, 0.5, 1, 2],
    "reg_alpha": [0, 0.01, 0.1, 0.5, 1]
}

best_params_per_month_xgb = {}

for month in range(1, 13):
    df = monthly_dfs_poly[month]

    X = df.drop(columns=["pr(mm)", "Date"], errors="ignore")
    y = df["pr(mm)"]

    X = pd.get_dummies(X, columns=["ClimateRegion"], prefix="Region")

    xgb = XGBRegressor(
        objective="reg:squarederror",
        random_state=42
    )

    search = RandomizedSearchCV(
        estimator=xgb,
        param_distributions=param_grid_xgb,
        n_iter=50,
        cv=tscv,
        scoring="r2",
        n_jobs=-1,
        random_state=42,
        verbose=0
    )

    search.fit(X, y)

    best_params_per_month_xgb[month] = search.best_params_

In [ ]:
def perform_walk_forward_xgb(df, training_window, forecast_horizon, best_params):
    train_r2s = []
    test_r2s = []

    for start in range(0, len(df) - training_window - forecast_horizon, forecast_horizon):
        train_df, test_df = split_data(df, start, training_window, forecast_horizon)

        X_train, X_test, y_train, y_test = prepare_features(train_df, test_df)

        model = XGBRegressor(
            objective="reg:squarederror",
            n_estimators=best_params["n_estimators"],
            max_depth=best_params["max_depth"],
            learning_rate=best_params["learning_rate"],
            subsample=best_params["subsample"],
            colsample_bytree=best_params["colsample_bytree"],
            gamma=best_params["gamma"],
            reg_alpha=best_params["reg_alpha"],
            random_state=42
        )

        model.fit(X_train, y_train)

        train_r2s.append(r2_score(y_train, model.predict(X_train)))
        test_r2s.append(r2_score(y_test, model.predict(X_test)))

    return train_r2s, test_r2s

In [ ]:
monthly_results_xgb = {}

for month in range(1, 13):
    config = best_configs[month]

    train_r2s, test_r2s = perform_walk_forward_xgb(
        monthly_dfs_poly[month],
        training_window=int(config["train_size"]),
        forecast_horizon=int(config["test_size"]),
        best_params=best_params_per_month_xgb[month]
    )

    monthly_results_xgb[month] = {
        "Train R²": np.mean(train_r2s),
        "Test R²": np.mean(test_r2s)
    }

xgb_results_summary = pd.DataFrame(monthly_results_xgb).T
xgb_results_summary

### XGBoost Results

XGBoost produced the strongest regression performance among the regression models tested. However, overall regression performance remained limited, which supported reframing the task as a classification problem in the next notebook.

## CatBoost Regression

CatBoost was tested as an additional gradient boosting model. Like Random Forest and XGBoost, it was tuned using randomized search with time-series cross-validation and evaluated with walk-forward validation.

In [ ]:
from catboost import CatBoostRegressor

param_grid_cat = {
    "iterations": [200, 400, 600],
    "depth": [4, 6, 8, 10],
    "learning_rate": [0.01, 0.05, 0.075, 0.1],
    "l2_leaf_reg": [1, 3, 5, 7, 10],
    "subsample": [0.6, 0.8, 1.0]
}

best_params_per_month_cat = {}

for month in range(1, 13):
    df = monthly_dfs_poly[month]

    X = df.drop(columns=["pr(mm)", "Date"], errors="ignore")
    y = df["pr(mm)"]

    X = pd.get_dummies(X, columns=["ClimateRegion"], prefix="Region")

    cat = CatBoostRegressor(
        random_state=42,
        verbose=0
    )

    search = RandomizedSearchCV(
        estimator=cat,
        param_distributions=param_grid_cat,
        n_iter=25,
        cv=tscv,
        scoring="r2",
        n_jobs=-1,
        random_state=42,
        verbose=0
    )

    search.fit(X, y)

    best_params_per_month_cat[month] = search.best_params_

In [ ]:
def perform_walk_forward_catboost(df, training_window, forecast_horizon, best_params):
    train_r2s = []
    test_r2s = []

    for start in range(0, len(df) - training_window - forecast_horizon, forecast_horizon):
        train_df, test_df = split_data(df, start, training_window, forecast_horizon)

        X_train, X_test, y_train, y_test = prepare_features(train_df, test_df)

        model = CatBoostRegressor(
            iterations=best_params["iterations"],
            learning_rate=best_params["learning_rate"],
            depth=best_params["depth"],
            l2_leaf_reg=best_params["l2_leaf_reg"],
            subsample=best_params["subsample"],
            random_state=42,
            verbose=0
        )

        model.fit(X_train, y_train)

        train_r2s.append(r2_score(y_train, model.predict(X_train)))
        test_r2s.append(r2_score(y_test, model.predict(X_test)))

    return train_r2s, test_r2s

In [ ]:
monthly_results_catboost = {}

for month in range(1, 13):
    config = best_configs[month]

    train_r2s, test_r2s = perform_walk_forward_catboost(
        monthly_dfs_poly[month],
        training_window=int(config["train_size"]),
        forecast_horizon=int(config["test_size"]),
        best_params=best_params_per_month_cat[month]
    )

    monthly_results_catboost[month] = {
        "Train R²": np.mean(train_r2s),
        "Test R²": np.mean(test_r2s)
    }

catboost_results_summary = pd.DataFrame(monthly_results_catboost).T
catboost_results_summary

### CatBoost Results

CatBoost was competitive with the other tree-based regression models, but it did not outperform the final XGBoost regression model. These results further supported the decision to compare regression with a classification-based approach.